# Causal Attribution Analysis

Evaluates PHANTOM's causal attribution accuracy against oracle ground truth.
Produces TABLE_2 and FIGURE_1 from the paper.

In [ ]:
# Cell 1: Setup and dataset load
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import json

DATASET_DIR = Path('research/datasets/phantom-v1')
RESULTS_DIR = Path('research/datasets/raw')

traces = pd.read_parquet(DATASET_DIR / 'traces.parquet')
labels = pd.read_parquet(DATASET_DIR / 'labels.parquet')

with open(DATASET_DIR / 'manifest.json') as f:
    manifest = json.load(f)

print(f"Traces: {len(traces):,} rows | Labels: {len(labels):,} rows")
print(f"PHANTOM version: {manifest['phantom_version']}")
print(f"Attack families: {labels[labels['is_attack']]['attack_family'].unique().tolist()}")


: 

In [ ]:
# Cell 2: Attribution accuracy computation
# Ground truth PURL vs PHANTOM top-attributed PURL
# Credit: 1.0 = exact match, 0.5 = oracle PURL in top-3, 0.0 = miss

attack_labels = labels[labels['is_attack']].copy()

def credit(row):
    truth = row.get('ground_truth_purl', '')
    top = row.get('top_attributed_purl', '')
    top3 = row.get('top3_attributed_purls', [])
    if isinstance(top3, str):
        try:
            top3 = json.loads(top3)
        except Exception:
            top3 = []
    if top and top == truth:
        return 1.0
    if truth in (top3 or []):
        return 0.5
    return 0.0

attack_labels['attribution_credit'] = attack_labels.apply(credit, axis=1)
exact_mask = attack_labels['attribution_credit'] == 1.0
partial_mask = attack_labels['attribution_credit'] == 0.5

print("Causal Attribution Accuracy (CAA)")
print(f"  Overall (mean credit): {attack_labels['attribution_credit'].mean():.3f}")
print(f"  Exact match rate:      {exact_mask.mean():.3f}")
print(f"  Partial credit rate:   {partial_mask.mean():.3f}")
print(f"  Zero credit rate:      {(attack_labels['attribution_credit'] == 0).mean():.3f}")
print()
print(attack_labels.groupby('attack_family')['attribution_credit'].describe().round(3))


In [ ]:
# Cell 3: Attribution confidence by scenario — box plot (FIGURE_1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: confidence distribution by attack family
conf_data = traces[
    (traces['label'] == 1) &
    (traces['phantom_attribution_confidence'].notna())
].copy()

families = conf_data['attack_family'].dropna().unique()
conf_by_family = [
    conf_data[conf_data['attack_family'] == f]['phantom_attribution_confidence'].values
    for f in families
]

axes[0].boxplot(conf_by_family, labels=[f.replace('_', '\n') for f in families], patch_artist=True)
axes[0].set_ylabel('Attribution Confidence')
axes[0].set_title('PHANTOM Attribution Confidence by Attack Family')
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='0.5 threshold')
axes[0].legend()

# Right: credit distribution
credit_counts = attack_labels['attribution_credit'].value_counts().sort_index()
axes[1].bar([str(c) for c in credit_counts.index], credit_counts.values,
            color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_xlabel('Attribution Credit')
axes[1].set_ylabel('Scenario Count')
axes[1].set_title('Attribution Credit Distribution\n(1.0=exact, 0.5=partial, 0.0=miss)')

plt.tight_layout()
plt.savefig('research/evaluation/results/figure_1_attribution_confidence.pdf', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


In [ ]:
# Cell 4: Identifiability analysis

# Load raw attribution results from saved JSON files
import glob

attribution_records = []
for f in sorted(glob.glob('research/datasets/raw/*.json')):
    if 'index' in f:
        continue
    try:
        with open(f) as fh:
            r = json.load(fh)
        label_dict = r.get('scenario_label', {})
        attribution_records.append({
            'run_id': r['run_id'],
            'attack_family': r['attack_family'],
            'ground_truth_label': r['ground_truth_label'],
            'identifiable': label_dict.get('identifiable'),
            'not_identifiable_reason': label_dict.get('not_identifiable_reason', ''),
        })
    except Exception:
        pass

adf = pd.DataFrame(attribution_records)
attack_adf = adf[adf['ground_truth_label'] == 1]

identifiable_counts = attack_adf['identifiable'].value_counts(dropna=False)
print("Identifiability outcomes (attack scenarios):")
print(identifiable_counts)

# Pie chart
fig, ax = plt.subplots(figsize=(6, 5))
labels_pie = identifiable_counts.index.astype(str).tolist()
ax.pie(identifiable_counts.values, labels=labels_pie, autopct='%1.1f%%',
       colors=['#2ecc71', '#e74c3c', '#95a5a6'])
ax.set_title('PHANTOM Causal Identifiability Outcomes\n(Attack Scenarios)')
plt.tight_layout()
plt.savefig('research/evaluation/results/figure_1b_identifiability.pdf', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 5: Refutation test stability
# Shows whether completed causal attributions pass the refutation tests

refutation_records = []
for f in sorted(glob.glob('research/datasets/raw/*.json')):
    if 'index' in f:
        continue
    try:
        with open(f) as fh:
            r = json.load(fh)
        label_dict = r.get('scenario_label', {})
        refutation_records.append({
            'run_id': r['run_id'],
            'attack_family': r['attack_family'],
            'ground_truth_label': r['ground_truth_label'],
            'refutation_passed': label_dict.get('refutation_passed'),
            'refutation_tests_run': label_dict.get('refutation_tests_run', 0),
        })
    except Exception:
        pass

rdf = pd.DataFrame(refutation_records)
attack_rdf = rdf[rdf['ground_truth_label'] == 1]

if 'refutation_passed' in attack_rdf.columns:
    refutation_rate = attack_rdf['refutation_passed'].mean()
    print(f"Refutation pass rate (attack scenarios): {refutation_rate:.3f}")
    print(attack_rdf.groupby('attack_family')['refutation_passed'].mean().round(3))
else:
    print("No refutation data in scenario labels yet (run with PHANTOM v2+).")
